In [1]:
import torch
from torch.utils.data import DataLoader
from transformers import T5TokenizerFast, CLIPProcessor


from peft import LoraConfig, get_peft_model, TaskType
from torch.amp import autocast, GradScaler
from tqdm import tqdm
import os 


In [2]:
from Modules.config import (TRAIN_IMAGE_DIR,
                            TEST_IMAGE_DIR,
                            FAISS_PATH,
                            TRAIN_METADATA_PATH,
                            TEST_METADATA_PATH,
                            CLIP_MODEL_NAME,
                            T5_MODEL_NAME,
                            CHECKPOINT_DIR)

from Modules.FusionVLM import FusionVLM
from Modules.retrieval_module import Retriever
from Modules.datasets import VLMDataset, VLMDataCollator
from Modules.utils import print_model_param_stats
from Modules.metrics import evaluate_captioning, setup_nltk

In [3]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE

'cuda'

In [4]:
CLIP_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_NAME, use_fast=True)
T5_tokenizer = T5TokenizerFast.from_pretrained(T5_MODEL_NAME)
collator = VLMDataCollator(CLIP_processor, T5_tokenizer, device=DEVICE)

'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 1332f04d-f6e6-4984-b861-377281842c81)')' thrown while requesting HEAD https://huggingface.co/openai/clip-vit-base-patch32/resolve/main/preprocessor_config.json
Retrying in 1s [Retry 1/5].


In [5]:
retriever = Retriever(metadata_path=TRAIN_METADATA_PATH, faiss_path=FAISS_PATH)

train_dataset = VLMDataset(image_dir=TRAIN_IMAGE_DIR, 
                           ref_image_dir=TRAIN_IMAGE_DIR,
                           metadata_path=TRAIN_METADATA_PATH,
                           retriever=retriever)

test_dataset = VLMDataset(image_dir=TEST_IMAGE_DIR, 
                           ref_image_dir=TRAIN_IMAGE_DIR,
                           metadata_path=TEST_METADATA_PATH,
                           retriever=retriever)

In [6]:
BATCH_SIZE = 16
NUM_WORKERS = 0

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, collate_fn=collator)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, collate_fn=collator)

In [ ]:
# import numpy as np
# from torch.utils.data import DataLoader, Subset

# fraction = 0.05
# num__train_samples = int(len(train_dataset) * fraction)
# num__test_samples = int(len(test_dataset) * fraction)

# indices = np.random.choice(len(train_dataset), num__train_samples, replace=False)
# train_subset = Subset(train_dataset, indices)
# train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, collate_fn=collator)

# indices = np.random.choice(len(test_dataset), num__test_samples, replace=False)
# test_subset = Subset(test_dataset, indices)
# test_loader = DataLoader(test_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, collate_fn=collator)

In [8]:
model = FusionVLM(
    vision_encoder_name="openai/clip-vit-base-patch32",
    text_encoder_name="t5-base",
    text_decoder_name="t5-base",
    num_fusion_blocks=4,
    use_local_files=True
).to(DEVICE)

num_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {num_params:,}")

Total parameters: 458,385,792


In [9]:
def freeze_module(module: torch.nn.Module):
    for p in module.parameters():
        p.requires_grad = False

# Freeze vision encoder
freeze_module(model.vision_encoder)

# Freeze text encoder
freeze_module(model.text_encoder)

In [10]:
lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=[
        # Decoder self-attention (language modeling)
        "SelfAttention.q",
        "SelfAttention.v",

        # Decoder cross-attention (fusion output → text)
        "EncDecAttention.q",
        "EncDecAttention.v",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM,
)

model.text_decoder = get_peft_model(model.text_decoder, lora_config)
model.text_decoder.print_trainable_parameters()


trainable params: 3,538,944 || all params: 226,442,496 || trainable%: 1.5628


In [11]:
# Unfreeze the lm head for token generation
for param in model.text_decoder.lm_head.parameters():
    print(param.shape[0]*param.shape[1])
    param.requires_grad = True

24674304


In [12]:
print_model_param_stats(model)

Module                                          Total    Trainable       Frozen
--------------------------------------------------------------------------------
vision_encoder                             87,456,000            0   87,456,000
text_encoder                              109,628,544            0  109,628,544
text_decoder                              226,442,496   28,213,248  198,229,248
vision_proj                                   590,592      590,592            0
fusion_blocks                              37,807,104   37,807,104            0
--------------------------------------------------------------------------------
TOTAL                                     461,924,736   66,610,944  395,313,792


In [13]:
NUM_EPOCHS = 1
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4,
    weight_decay=0.01
)

setup_nltk()
scaler = GradScaler()

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Mahan\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Mahan\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [14]:
def evaluate_epoch(model, dataloader, tokenizer, device=DEVICE):
    model.eval()

    preds = []
    refs = []

    with torch.no_grad():
        for batch in dataloader:
            gt_captions = batch["all_captions"]  # List[List[str]]
            
            generated_ids = model.generate(
                    query_pixel_values=batch["query_pixel_values"],
                    retrieved_pixel_values=batch["retrieved_pixel_values"],
                    input_ids=batch["input_ids"],
                    attention_mask=batch["attention_mask"],
                    max_length=64,
                    num_beams=4
            )

            decoded = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
            preds.extend(decoded)
            refs.extend(gt_captions)

    return evaluate_captioning(preds, refs)

In [15]:
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

loss_history = []

for epoch in range(NUM_EPOCHS):
    model.train()
    batch_loss = 0.0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}", leave=True)

    for batch in progress_bar:
        optimizer.zero_grad()

        with autocast(device_type="cuda", dtype=torch.float16):
            outputs = model(
                query_pixel_values=batch["query_pixel_values"],
                retrieved_pixel_values=batch["retrieved_pixel_values"],
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                labels=batch["labels"]
            )
            loss = outputs.loss

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        batch_loss += loss.item()
        # progress_bar.set_postfix(loss=loss.item())

    epoch_loss = batch_loss / len(train_loader)
    loss_history.append(epoch_loss)
    
    progress_bar.set_postfix(final_loss=epoch_loss)
    print(f"Epoch {epoch+1} Avg Loss: {epoch_loss:.4f}")

    # Evaluate
    metrics = evaluate_epoch(model, test_loader, T5_tokenizer, DEVICE)
    # print(f"\nEpoch {epoch}")
    for k, v in metrics.items():
        print(f"{k}: {v:.4f}")


    # ---- Backup every 5 epochs ----
    if (epoch + 1) % 5 == 0:
        backup_path = os.path.join(CHECKPOINT_DIR, f"FusionVLM_epoch{epoch+1}.pt")
        torch.save(model.state_dict(), backup_path)


Epoch 1: 100%|██████████| 97/97 [00:47<00:00,  2.02it/s]


Epoch 1 Avg Loss: 3.5373
BLEU-1: 0.0000
BLEU-2: 0.0000
BLEU-3: 0.0000
BLEU-4: 0.0000
METEOR: 0.0000
ROUGE-L: 0.0000
CIDEr: 0.0000
